# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mansi-1002/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, subprocess
import pandas as pd, numpy as np

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(path):
    path = f"{REPO_DIR}/{path}"
    if not os.path.exists(path):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

df = pd.read_csv(path)
# Used ONLY to check signals and review picks. Never used as a score input.
df["declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape, "| declining rate:", round(df["declining"].mean(), 3))

(30000, 45) | declining rate: 0.542


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
df["stale_bucket"] = pd.cut(df["days_since_last_update"], [-1, 90, 180, 365, 10**6],
                            labels=["0-90", "91-180", "181-365", "365+"])
print("SIGNAL 1: staleness (behind the refresh flags)")
print(df.groupby("stale_bucket", observed=True)["declining"].agg(n="size", declining_rate="mean").round(3))

print("\nSIGNAL 2: position tier")
print(df.groupby("position_tier", observed=True)["declining"].agg(n="size", declining_rate="mean").round(3))

SIGNAL 1: staleness (behind the refresh flags)
                  n  declining_rate
stale_bucket                       
0-90          20655           0.512
91-180         9171           0.611
181-365         169           0.467
365+              5           0.600

SIGNAL 2: position tier
                   n  declining_rate
position_tier                       
deep            1319           0.344
page_1         11814           0.570
page_3_5        7242           0.562
striking        7304           0.610
top_3           2321           0.241


**Signal checks (observed, directional):**
- Staleness: **[VERDICT]**. The declining rate goes from [x] in the freshest bucket
  (n=[n]) to [x] in the stalest bucket (n=[n]). [One sentence on what that means.]
- Position tier: **[VERDICT]**. [One sentence using the rates and n.]

**My rule in plain words:** rank pages that are stale (180+ days since the last update)
and still visible (impressions_90d > 0). The more exposure a stale page has, the higher
it ranks, because more traffic is at risk.

**Score:** impressions_90d for pages that are 180+ days stale, otherwise 0.
**Reason code:** STALE_VISIBLE (stale and visible), or NO_ACTION.
**Action label:** "Review and refresh content" for STALE_VISIBLE, "Leave as is" otherwise.
**Inputs:** only days_since_last_update and impressions_90d, both known before the decision.

## 2. Build the ranked queue (writes the CSV)

In [5]:
df["score"] = (df["days_since_last_update"] >= 180).astype(int) * df["impressions_90d"]
df["reason_code"] = np.where(df["score"] > 0, "STALE_VISIBLE", "NO_ACTION")
df["action"] = np.where(df["score"] > 0, "Review and refresh content", "Leave as is")

queue = df.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["rank", "score", "reason_code", "action",
            "impressions_90d", "days_since_last_update", "avg_position"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

p20 = queue.head(20)["declining"].mean()
p50 = queue.head(50)["declining"].mean()
print(f"Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")
print(f"Baseline: top-20 declining rate {p20:.3f}, top-50 declining rate {p50:.3f}")

Wrote 30,000 rows to work/outputs/baseline_action_score.csv
Baseline: top-20 declining rate 0.900, top-50 declining rate 0.680


In [6]:
import json
json.dump({"baseline_top20": float(p20), "baseline_top50": float(p50)},
          open("work/outputs/baseline_metrics.json", "w"))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
top20 = queue.head(20)
for _, r in top20.iterrows():
    print(f"#{int(r['rank'])}: {r['action']} | {r['reason_code']} | "
          f"{int(r['days_since_last_update'])} days stale | "
          f"{int(r['impressions_90d'])} impressions | avg position {r['avg_position']:.1f}")

#1: Review and refresh content | STALE_VISIBLE | 194 days stale | 61678 impressions | avg position 19.7
#2: Review and refresh content | STALE_VISIBLE | 194 days stale | 59472 impressions | avg position 24.8
#3: Review and refresh content | STALE_VISIBLE | 194 days stale | 25715 impressions | avg position 22.2
#4: Review and refresh content | STALE_VISIBLE | 193 days stale | 13299 impressions | avg position 10.5
#5: Review and refresh content | STALE_VISIBLE | 194 days stale | 7812 impressions | avg position 39.0
#6: Review and refresh content | STALE_VISIBLE | 193 days stale | 7558 impressions | avg position 17.9
#7: Review and refresh content | STALE_VISIBLE | 194 days stale | 4590 impressions | avg position 31.0
#8: Review and refresh content | STALE_VISIBLE | 194 days stale | 4556 impressions | avg position 16.4
#9: Review and refresh content | STALE_VISIBLE | 194 days stale | 4429 impressions | avg position 25.3
#10: Review and refresh content | STALE_VISIBLE | 193 days stale | 16

1. Review and refresh | STALE_VISIBLE | confidence: medium (high exposure but one signal only)
   Wrong if: the traffic is seasonal, or the page was recently updated in a way the
   data does not capture.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
score_inputs = ["days_since_last_update", "impressions_90d"]
forbidden = ["trend_direction", "trend_pct", "declining"]
leaked = [c for c in score_inputs if c in forbidden]
print("Score inputs:", score_inputs)
print("Forbidden columns used as inputs:", leaked or "none")
assert not leaked

Score inputs: ['days_since_last_update', 'impressions_90d']
Forbidden columns used as inputs: none


**Weak picks:**
- [Rank #x]: [what looks wrong and why, e.g. very high impressions on an evergreen page].
- [Rank #y]: [another weak pick].
These suggest my rule over-ranks [pattern], so my Week 5 model should [one idea].

**Leakage check:** the score uses only days_since_last_update and impressions_90d. The label
(trend_direction) and trend_pct are not inputs. No product flags and no future-window
columns were used. trend_direction was used only afterwards to judge the picks.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.